# 7장 — 반쪽 자: fp16·bf16과 혼합 정밀도 (핵심) ✂️

웹 챕터의 세 실험(세 자 대조 · 누적 분업 · loss scaling)을 numpy 로 재현합니다.
(bf16 은 numpy 에 없어서 **fp32 비트를 직접 잘라** 만들어 봅니다. 2장 view 재주의 실전!)

> **노트북 사용법**: 각 셀을 클릭하고 `Shift + Enter`를 누르며 위에서 아래로 내려가세요.

**이 노트북의 목표**: 혼합 정밀도의 핵심.
**"저장·곱셈은 반쪽 자로, 티끌이 쌓이는 곳은 fp32 로."**

In [ ]:
# 🤖 AI 튜터 — 공부하다 막히면 오른쪽 아래 버튼을 눌러 물어보세요 (로컬 Ollama, 없어도 학습 지장 없음)
from tutor import chat
chat('ch7')

## 1. bf16 을 손수 만들기: fp32 의 위쪽 16비트

numpy 에는 float16 은 있지만 bfloat16 이 없습니다. 마침 잘 됐죠. bf16 의 정의가
"fp32 의 위 16비트"라는 걸 이용해 **직접 만들며** 구조를 확인합니다.

In [ ]:
import numpy as np
from decimal import Decimal

def to_bf16(x):
    """bf16 = fp32 비트의 아래 16비트를 반올림(RNE)해 잘라낸 것."""
    u = np.float32(x).view(np.uint32)
    lsb = (u >> np.uint32(16)) & np.uint32(1)
    u = (u + np.uint32(0x7FFF) + lsb) & np.uint32(0xFFFF0000)
    return float(u.view(np.float32))

print('0.1 을 세 자에 담으면:')
print(f'  fp32: {Decimal(float(np.float32(0.1)))}')
print(f'  fp16: {Decimal(float(np.float16(0.1)))}')
print(f'  bf16: {Decimal(to_bf16(0.1))}')

assert float(np.float16(0.1)) == 0.0999755859375, 'fp16(0.1) — 3장 티저의 그 값 (웹 대조기와 일치)'
assert to_bf16(0.1) == 0.10009765625, 'bf16(0.1) — 웹 대조기와 비트까지 일치'
assert to_bf16(70000.0) != float('inf') and np.isinf(np.float16(70000)), '70000: fp16 은 inf(최대 65,504), bf16 은 멀쩡'
assert np.float16(2e-8) == 0.0 and to_bf16(2e-8) > 0, '2×10⁻⁸: fp16 은 소실, bf16 은 생존'
assert np.isinf(np.float16(65520)) and float(np.float16(65504)) == 65504.0, 'fp16 자의 끝 = 65,504'
print()
print('✔ 확인: 같은 16비트, 다른 배분. fp16 은 눈금(가수 10), bf16 은 자 길이(지수 8 = fp32)')

## 2. 눈금 간격 대조: 1 근처에서 세 자의 촘촘함

In [ ]:
def bf16_spacing_at_1():
    # 1.0 의 다음 bf16 눈금 = 가수 7비트 → 1 + 2⁻⁷
    return to_bf16(1.0 + 2**-7) - 1.0

print(f'1 근처 눈금 간격, fp32: {np.spacing(np.float32(1.0)):.3e}')
print(f'                  fp16: {np.spacing(np.float16(1.0)):.3e}')
print(f'                  bf16: {bf16_spacing_at_1():.3e}')

assert np.spacing(np.float16(1.0)) == 2.0**-10, 'fp16: 2⁻¹⁰ ≈ 0.00098 — 실험 ②(흡수)의 복선'
assert bf16_spacing_at_1() == 2.0**-7, 'bf16: 2⁻⁷ ≈ 0.0078 — 셋 중 가장 성긴 눈금'
print()
print('✔ 확인: bf16 은 fp16 보다도 눈금이 8배 성글다. 그런데도 훈련 기본값이 된 이유는 3절에서')

## 3. 누적 분업 실험: "저장은 반쪽, 누적은 fp32"

1.0 에 0.0001 을 3000번 더합니다. 순수 fp16 누적 vs 혼합(저장 fp16 · 누적 fp32).

In [ ]:
N, TINY = 3000, np.float16(0.0001)

pure = np.float16(1.0)
for _ in range(N):                     # 순수 fp16 — 누적기도 반쪽 자
    pure = np.float16(pure + TINY)

mixed = np.float32(1.0)
for _ in range(N):                     # 혼합 — 같은 fp16 값을 fp32 누적기로
    mixed = np.float32(mixed + np.float32(TINY))

exact = 1 + N * 0.0001
print(f'순수 fp16 누적 = {float(pure)}   (0.0001 이 눈금 절반 0.00049 보다 작아 매번 흡수)')
print(f'혼합 누적      = {float(mixed):.4f}')
print(f'참값           = {exact}')

assert float(pure) == 1.0, '순수 fp16: 3000번을 더해도 1.0 — 4장 흡수의 재공연'
assert abs(float(mixed) - exact) < 1e-3, '혼합: 저장은 같은 fp16 값인데 누적기만 바꿔도 1.3 도달'
print()
print('✔ 확인: 혼합 정밀도의 핵심. 텐서 코어도 fp16 곱셈을 내부에선 fp32 로 누적한다')

## 4. loss scaling 구급차: 소실과 오버플로 사이

작은 gradient(그래디언트, 우리말로 **기울기**. 2부 9장의 그 학습 신호, 크기 2×10⁻⁸)를
×S 확대 → fp16 저장 → ÷S 복원. S 의 양쪽 실패까지.

In [ ]:
g = 2e-8                                 # fp16 아래끝(≈6×10⁻⁸)보다 작은 gradient

dead = float(np.float16(g))              # 그대로 저장 → 소실
S = 1024
rescued = float(np.float16(g * S)) / S   # 확대 → 저장 → 복원

print(f'그대로 저장     : {dead}   (학습 신호 소멸!)')
print(f'S=1024 로 구출  : {rescued:.3e}   (참값 {g:.0e}, 오차 {abs(rescued - g) / g * 100:.1f}%)')

assert dead == 0.0, '소실 — loss scaling 이 필요한 이유'
assert abs(rescued - g) / g < 0.05, '×1024 → fp16 → ÷1024 로 5% 안에 회복'
assert np.isinf(np.float16(1.0 * 65536)), 'S 과대의 실패 — 큰 gradient(1.0)에 S=65536 이면 위로 터짐'
print()
print('✔ 확인: S 도 다이얼. 과소는 소실, 과대는 오버플로 → 실전은 동적(dynamic) loss scaling')

## 5. 세 자의 지도: 눈금 간격을 겹쳐 그리기

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

xs = 10.0 ** np.arange(-7, 5)
g32 = np.spacing(xs.astype(np.float32)).astype(float)
g16 = [float(np.spacing(np.float16(x))) if np.isfinite(np.float16(x)) and np.float16(x) > 0 else np.nan for x in xs]
gbf = [to_bf16(x * (1 + 2**-7)) - to_bf16(x) for x in xs]   # bf16 이웃 간격(근사 실측)

plt.figure(figsize=(6.5, 3.8))
plt.loglog(xs, g16, 'o-', color='tab:red', label='fp16 (가수 10): 3.4만 너머는 자가 없다')
plt.loglog(xs, gbf, 's-', color='tab:orange', label='bf16 (가수 7): 성글지만 자는 길다')
plt.loglog(xs, g32, '^-', color='tab:gray', label='fp32 (가수 23)')
plt.axhline(1, color='tab:red', ls=':', lw=1)
plt.xlabel('수의 크기'); plt.ylabel('이웃 눈금까지의 간격'); plt.legend(fontsize=8); plt.grid(alpha=0.3, which='both')
plt.title('세 고무줄 자: 높이(밀도)가 다르고, fp16 은 길이도 짧다'); plt.tight_layout(); plt.show()
print('✔ 확인: 배포(추론)는 반쪽 자로 충분, 훈련은 분업(혼합 정밀도). 다음 장부터는 정수 자(양자화)')

**직접 해보기**: ① `np.finfo(np.float16)` 을 출력해 max·tiny 를 확인해 보세요.
② s4 의 S 를 65536 으로 바꾸면 g=2e-8 은 어떻게 되나요? (힌트: 2e-8×65536 ≈ 0.0013, 잘 담기고
복원도 됩니다. 문제는 같은 S 를 **큰 gradient 도 함께** 통과한다는 것. 동적 조절의 이유죠.)

## 6. 관찰 정리 + 🤖 AI 연결

- **fp16(1+5+10)**: 눈금 우선, 자 짧음(65,504·아래끝 6×10⁻⁸) → 오버플로·소실 위험.
  **bf16(1+8+7)**: 자 = fp32, 눈금 성김 → 폭발 걱정 없음. **훈련 기본값 = bf16**
  (오버플로는 즉사, 정밀도 손실은 관리 가능). 설정 파일의 `bf16: true`.
- **혼합 정밀도** = 분업: 저장·곱셈 반쪽 자 / **누적기·master weights·옵티마이저 fp32**
  (순수 fp16 누적은 1.0 에서 멈췄다, 흡수).
- **loss scaling**: ×S → fp16 backward → ÷S. 과소 = 소실, 과대 = 오버플로 → 동적 조절.
- 훈련 메모리는 파라미터당 ≈16B(fp16 2 + fp32 원본 4 + Adam 8 + grad 2). 추론의 8배 급.
- **다시 쓰는 수학**: 3장(비트 배분 = 범위 vs 정밀도), 4장(흡수), 5장(자의 끝),
  8부 9장(사고 처방 목록에 합류), 4부 11장(학습 루프의 어디가 반쪽인가).

**다음 챕터**: 8장, 눈금 갈아타기(양자화). 실수 자를 버리고 int8 정수 자 + 축척 스티커로 옮겨 가며
4GB 파일의 비밀이 시작됩니다.